evaluation data is converted to Dataframe to check the overall acurracy and give a detailed report about of what questions it is not working properly. 

In [6]:
import pandas as pd
import json
import os


In [38]:
path = "D:\Kenpath\oan-evaluation"

In [39]:
os.chdir(path)

In [40]:
# Path to the data file
json_path = r"data/models/mistralai_Mistral-Small-3.2-24B-Instruct-2506/evaluation.json"

In [41]:
# Open and load the JSON data
with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Convert to DataFrame - handle if it's an array of dicts, or a dict of dicts
if isinstance(data, dict):
    # Try to convert dict of items to list of dicts
    df = pd.DataFrame(list(data.values()))
else:
    df = pd.DataFrame(data)

In [42]:
df

,question,category,agristack_required,farmer_id,answer,agent_turns,evaluation
0,माझा महाडीबीटी अर्ज स्टेटस दाखवा.,MahaDBT,Yes,50396788689,**महाडीबीटी अर्ज स्थिती:**\n\nआपल्या प्रोफाइलम...,[{'parts': [{'tool_name': 'fetch_agristack_dat...,"{'protocol': {'scope_discipline': {'score': 5,..."
1,Aadhaar eKYC MahaDBT madhe yashaswi jhale ka?,MahaDBT,Yes,30230052162,**Assistant:**\n\nआधार ईकायसी (eKYC) महाडीबीटी...,"[{'parts': [{'tool_name': 'get_scheme_codes', ...","{'protocol': {'scope_discipline': {'score': 5,..."
2,MahaDBT पोर्टलवर अर्जाची पडताळणी पूर्ण होण्यास...,MahaDBT,Yes,26815933009,**Assistant:**\n\nमहाराष्ट्रातील विविध शासकीय ...,"[{'parts': [{'tool_name': 'get_scheme_codes', ...","{'protocol': {'scope_discipline': {'score': 5,..."
3,बँक खाते पडताळणी अयशस्वी झाल्यास महाडीबीटीमध्य...,MahaDBT,Yes,15759514812,**Assistant:**\n\nमहाराष्ट्र शासनाच्या विविध क...,"[{'parts': [{'tool_name': 'get_scheme_codes', ...","{'protocol': {'scope_discipline': {'score': 5,..."
4,MahaDBT लाभ कोणत्या बँकेत जमा झाला आहे ते तपासा.,MahaDBT,Yes,62891426784,**Assistant:**\n\n📊 **आपल्या माहाडीबीटी अर्जाच...,[{'parts': [{'tool_name': 'fetch_agristack_dat...,"{'protocol': {'scope_discipline': {'score': 5,..."
...,...,...,...,...,...,...,...
96,पुढील ३ दिवस माझ्या क्षेत्रातील हवामान कापूस व...,Weather Forecast,Yes,50396788689,"**Weather Forecast for Renapur, Latur (Next 3 ...",[{'parts': [{'tool_name': 'fetch_agristack_dat...,"{'protocol': {'scope_discipline': {'score': 5,..."
97,पुढील आठवड्यात होणाऱ्या पावसाचा सोयाबीन मळणीवर...,Weather Forecast,Yes,50396788689,"**Weather Forecast for Renapur, Latur (6th Jan...",[{'parts': [{'tool_name': 'fetch_agristack_dat...,"{'protocol': {'scope_discipline': {'score': 5,..."
98,वाढत्या तापमानामुळे संत्रा बागेचे नुकसान टाळण्...,Weather Forecast,Yes,26815933009,**संत्रा बागेतील तापमान ताणाचे नियंत्रण (Tempe...,"[{'parts': [{'tool_name': 'search_terms', 'arg...","{'protocol': {'scope_discipline': {'score': 5,..."
99,तासवार हवामान अंदाज आणि पावसाची शक्यता तपासून ...,Weather Forecast,Yes,50396788689,"**Weather Forecast for Renapur, Latur (06 Janu...",[{'parts': [{'tool_name': 'fetch_agristack_dat...,"{'protocol': {'scope_discipline': {'score': 5,..."


In [43]:
# Parse and normalize evaluation column
evals = df['evaluation'].apply(lambda x: json.loads(x) if isinstance(x, str) else (x if isinstance(x, dict) else {}))
eval_df = pd.json_normalize(evals)

# Join back to original dataframe
df_with_eval = pd.concat([df.drop(columns=['evaluation']), eval_df], axis=1)
df_with_eval

,question,category,agristack_required,farmer_id,answer,agent_turns,overall_score,summary,protocol.scope_discipline.score,protocol.scope_discipline.explanation,...,response.conversation_closure_quality.evidence,integrity.tool_name_confidentiality.score,integrity.tool_name_confidentiality.explanation,integrity.tool_name_confidentiality.evidence,marathi.marathi_linguistic_quality.score,marathi.marathi_linguistic_quality.explanation,marathi.marathi_linguistic_quality.evidence,marathi.marathi_terminology_fidelity.score,marathi.marathi_terminology_fidelity.explanation,marathi.marathi_terminology_fidelity.evidence
0,माझा महाडीबीटी अर्ज स्टेटस दाखवा.,MahaDBT,Yes,50396788689,**महाडीबीटी अर्ज स्थिती:**\n\nआपल्या प्रोफाइलम...,[{'parts': [{'tool_name': 'fetch_agristack_dat...,9,"Strongest: Perfect tool workflow and faithful,...",5,Stayed within MahaDBT status-check scope; no o...,...,None,5,No internal tool names exposed in the final an...,None,5,Correct grammar and natural phrasing; no code-...,"[""आपल्या प्रोफाइलमध्ये कोणतेही योजना अर्ज सापड...",5,Uses appropriate Marathi terms for the context...,"[""महाडीबीटी अर्ज स्थिती"", ""योजना अर्ज""]"
1,Aadhaar eKYC MahaDBT madhe yashaswi jhale ka?,MahaDBT,Yes,30230052162,**Assistant:**\n\nआधार ईकायसी (eKYC) महाडीबीटी...,"[{'parts': [{'tool_name': 'get_scheme_codes', ...",2,Stayed on-topic and did not reveal tools. Majo...,5,Query was MahaDBT-specific and the assistant s...,...,[Ends with 'संपर्क साधा...' but no specific qu...,5,The final user-facing answer did not reveal in...,[No mention of get_scheme_codes/get_scheme_inf...,1,Frequent grammatical errors and unnatural cons...,"['कrokड्', 'मोदीफाई प्रोफाइल', inconsistent ca...",1,Heavy code-switching and incorrect/irrelevant ...,[Use of English terms within Marathi sentences...
2,MahaDBT पोर्टलवर अर्जाची पडताळणी पूर्ण होण्यास...,MahaDBT,Yes,26815933009,**Assistant:**\n\nमहाराष्ट्रातील विविध शासकीय ...,"[{'parts': [{'tool_name': 'get_scheme_codes', ...",3,Clear Marathi tone and no tool-name leakage. M...,5,Stuck to a MahaDBT (agri schemes) query and di...,...,None,5,Did not mention internal tool names in the fin...,None,4,Generally correct Marathi with clear phrasing;...,None,3,"Used basic Marathi terms (उदा., पडताळणी) but r...",None
3,बँक खाते पडताळणी अयशस्वी झाल्यास महाडीबीटीमध्य...,MahaDBT,Yes,15759514812,**Assistant:**\n\nमहाराष्ट्र शासनाच्या विविध क...,"[{'parts': [{'tool_name': 'get_scheme_codes', ...",2,Stayed in scope and avoided tool-name leakage....,5,The reply stays within the MahaDBT/government ...,...,"[Final line: 'आपण कोणतेही प्रश्न असल्यास, कृपय...",5,No internal tool names or system details were ...,[Final Response contains no tool identifiers.],1,Grammar and structure are poor with heavy redu...,[Multiple lines repeat awkward constructs: 'अर...,1,Frequent English–Marathi code-switching for ke...,"[Uses 'Apply for Scheme', 'Application Status'..."
4,MahaDBT लाभ कोणत्या बँकेत जमा झाला आहे ते तपासा.,MahaDBT,Yes,62891426784,**Assistant:**\n\n📊 **आपल्या माहाडीबीटी अर्जाच...,[{'parts': [{'tool_name': 'fetch_agristack_dat...,5,Best points: Correct tool workflow (Agristack ...,5,Stayed within MahaDBT scheme/status context fo...,...,[Final: 'आपणाला इतर कोणतीही मदत हवी आहे का?'],5,No internal tool names or processes were expos...,[Final contains no references to 'fetch_agrist...,3,"Mostly correct Marathi, but with a few grammat...","[Inconsistent 'माहाडीबीटी' vs 'महाडीबीटी', Phr...",3,Some terms are off or non-standard for schemes...,['संस्थान: 2' used where 'घटक: 2' would be sta...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,पुढील ३ दिवस माझ्या क्षेत्रातील हवामान कापूस व...,Weather Forecast,Yes,50396788689,"**Weather Forecast for Renapur, Latur (Next 3 ...",[{'parts': [{'tool_name': 'fetch_agristack_dat...,6,Strong process: correctly used Agristack locat...,5,Stayed within agricultural weather advisory sc...,...,"[""कापूस वेचणीसाठी कोणते इतर उपाय आपण विचारू इच...",5,No internal tool names or

In [44]:
# Extract class names (top-level keys in eval_df that end with '.score')
score_cols = [col for col in df_with_eval.columns if col.endswith('.score')]

# Select required columns and score columns
selected_columns = [
    'question', 'category', 'agristack_required', 'farmer_id', 'answer',
    'overall_score', 'summary'
] + score_cols

# Handle: some columns may be missing if data format is inconsistent
cols_present = [col for col in selected_columns if col in df_with_eval.columns]

filtered_df = df_with_eval[cols_present]

In [52]:
# Filter rows with overall_score less than 6
final_df = filtered_df[filtered_df['overall_score'] < 8]

In [53]:
final_df.to_csv("final_df.csv", index=False)